# Qualitative Analiz & Görselleştirme

**Gereksinimler (Drive'da olması gereken):**
- `MyDrive/stgcn/processed_xsub.npz`
- `MyDrive/stgcn/outputs/best_model.pth` (02_model_training.ipynb çıktısı)

Bu notebook ablasyon sonuçlarına ihtiyaç duymaz — sadece en iyi modeli kullanır.

## 0. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pillow

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyArrowPatch
from IPython.display import Image, display
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DRIVE_ROOT  = '/content/drive/MyDrive/stgcn'
NPZ_PATH    = os.path.join(DRIVE_ROOT, 'processed_xsub.npz')
CKPT_PATH   = os.path.join(DRIVE_ROOT, 'outputs/best_model.pth')
VIZ_DIR     = os.path.join(DRIVE_ROOT, 'visualizations')
os.makedirs(VIZ_DIR, exist_ok=True)

## 1. Model & Veri Yükle

In [ ]:
# ── Sabitler ───────────────────────────────────────────────────────────────
COCO_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
NUM_JOINTS = 17

NTU60_CLASSES = [
    'drink water', 'eat meal/snack', 'brushing teeth', 'brushing hair', 'drop',
    'pickup', 'throw', 'sitting down', 'standing up', 'clapping',
    'reading', 'writing', 'tear up paper', 'wear jacket', 'take off jacket',
    'wear a shoe', 'take off a shoe', 'wear on glasses', 'take off glasses',
    'put on a hat/cap', 'take off a hat/cap', 'cheer up', 'hand waving',
    'kicking something', 'reach into pocket', 'hopping (one foot jumping)',
    'jump up', 'make a phone call', 'playing with phone/tablet',
    'typing on a keyboard', 'pointing to something', 'taking a selfie',
    'check time (from watch)', 'rub two hands together', 'nod head/bow',
    'shake head', 'wipe face', 'salute', 'put the palms together',
    'cross hands in front', 'sneeze/cough', 'staggering', 'falling',
    'touch head (headache)', 'touch chest (stomachache)', 'touch back (backache)',
    'touch neck (neckache)', 'nausea or vomiting', 'use a fan/feeling warm',
    'punching other person', 'kicking other person', 'pushing other person',
    'pat on back of other', 'point finger at other', 'hugging other person',
    'giving something to other', 'touch others pocket',
    'handshaking', 'walking towards each other', 'walking apart'
]

# Joint renkleri: baş=pembe, sol kol=mavi, sağ kol=kırmızı, gövde=yeşil, bacaklar=turuncu
JOINT_COLORS = [
    '#FF69B4',  # 0 nose
    '#FF69B4', '#FF69B4',  # 1-2 eyes
    '#FF69B4', '#FF69B4',  # 3-4 ears
    '#4169E1', '#DC143C',  # 5-6 shoulders
    '#4169E1', '#DC143C',  # 7-8 elbows
    '#4169E1', '#DC143C',  # 9-10 wrists
    '#228B22', '#228B22',  # 11-12 hips
    '#FF8C00', '#FF8C00',  # 13-14 knees
    '#FF8C00', '#FF8C00',  # 15-16 ankles
]

EDGE_COLORS = {
    (0,1): '#FF69B4', (0,2): '#FF69B4', (1,3): '#FF69B4', (2,4): '#FF69B4',
    (5,6): '#32CD32', (5,7): '#4169E1', (7,9): '#4169E1',
    (6,8): '#DC143C', (8,10): '#DC143C',
    (5,11): '#228B22', (6,12): '#228B22', (11,12): '#228B22',
    (11,13): '#FF8C00', (13,15): '#FF8C00',
    (12,14): '#FF8C00', (14,16): '#FF8C00',
}

In [ ]:
# ── Model tanımı (02_model_training.ipynb ile aynı) ────────────────────────

def build_adj(num_joints=NUM_JOINTS, pairs=COCO_PAIRS):
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in pairs:
        A[i, j] = A[j, i] = 1.0
    A += np.eye(num_joints, dtype=np.float32)
    D = np.diag(A.sum(1) ** -0.5)
    return torch.FloatTensor(D @ A @ D)

class SpatialGCN(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.bn   = nn.BatchNorm2d(out_c)
    def forward(self, x):
        return self.bn(self.conv(torch.einsum('nctv,vw->nctw', x, self.A)))

class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1):
        super().__init__()
        self.gcn = SpatialGCN(in_c, out_c, A)
        self.tcn = nn.Sequential(
            nn.Conv2d(out_c, out_c, (9,1), stride=(stride,1), padding=(4,0)),
            nn.BatchNorm2d(out_c),
        )
        self.residual = (
            nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=(stride,1)), nn.BatchNorm2d(out_c))
            if in_c != out_c or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.tcn(self.relu(self.gcn(x))) + self.residual(x))

class STGCN(nn.Module):
    def __init__(self, num_classes=60, in_channels=2, A=None):
        super().__init__()
        A = A if A is not None else build_adj()
        self.data_bn = nn.BatchNorm1d(in_channels * NUM_JOINTS)
        self.layers  = nn.ModuleList([
            STGCNBlock(in_channels, 64,  A),
            STGCNBlock(64,  64,  A), STGCNBlock(64, 64, A), STGCNBlock(64, 64, A),
            STGCNBlock(64,  128, A, stride=2),
            STGCNBlock(128, 128, A), STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2),
            STGCNBlock(256, 256, A), STGCNBlock(256, 256, A),
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.5)
        self.fc   = nn.Linear(256, num_classes)

    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.permute(0,4,1,3,2).contiguous().view(N*M, C*V, T)
        x = self.data_bn(x)
        x = x.view(N*M, C, V, T).permute(0,1,3,2)
        for layer in self.layers:
            x = layer(x)
        x = self.pool(x).view(N*M, -1)
        return self.fc(self.drop(x.view(N, M, -1).mean(1)))

In [ ]:
# Model yükle
A = build_adj()
model = STGCN(num_classes=60, A=A).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Model yüklendi — epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']*100:.2f}%")

# Val veriyi yükle — ham (M, T, V, C) formatında sakla (animasyon için)
d = np.load(NPZ_PATH)
X_val, y_val = d['X_val'], d['y_val']   # (N, M, T, V, C)
print(f"Val set: {X_val.shape}  labels: {y_val.shape}")

In [ ]:
# Tüm val tahminlerini al
def get_predictions(model, X, batch=128):
    all_preds, all_probs = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.FloatTensor(X[i:i+batch])   # (B, M, T, V, C)
            xb = xb.permute(0,4,2,3,1).to(device)  # (B, C, T, V, M)
            with autocast():
                logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_preds.append(probs.argmax(1))
            all_probs.append(probs)
    return np.concatenate(all_preds), np.concatenate(all_probs)

print("Tahminler hesaplanıyor...")
preds, probs = get_predictions(model, X_val)
correct_mask = (preds == y_val)
top1 = correct_mask.mean() * 100
print(f"Val Top-1 Accuracy: {top1:.2f}%")

## 2. Per-Class Accuracy

In [ ]:
per_class_acc = np.zeros(60)
for c in range(60):
    mask = y_val == c
    if mask.sum() > 0:
        per_class_acc[c] = (preds[mask] == c).mean()

# Renk: yeşil=iyi, kırmızı=kötü
colors = ['#2ecc71' if a >= 0.85 else '#e74c3c' if a < 0.60 else '#f39c12'
          for a in per_class_acc]

fig, ax = plt.subplots(figsize=(20, 6))
bars = ax.bar(range(60), per_class_acc * 100, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(top1, color='black', linestyle='--', linewidth=1.5, label=f'Ortalama={top1:.1f}%')
ax.axhline(85, color='gray', linestyle=':', linewidth=1, label='Hedef=85%')
ax.set_xticks(range(60))
ax.set_xticklabels([f"{i}" for i in range(60)], fontsize=7)
ax.set_xlabel('Action Class ID')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy — NTU60 XSub Val Set', fontsize=13)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='≥85%'),
    Patch(facecolor='#f39c12', label='60–85%'),
    Patch(facecolor='#e74c3c', label='<60%'),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][2:],
          loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'per_class_accuracy.png'), dpi=130)
plt.show()

print(f"\nGreen (≥85%): {(per_class_acc>=0.85).sum()} class")
print(f"Orange (60-85%): {((per_class_acc>=0.60)&(per_class_acc<0.85)).sum()} class")
print(f"Red (<60%): {(per_class_acc<0.60).sum()} class")

In [ ]:
# En iyi ve en kötü 10 class
sorted_idx = np.argsort(per_class_acc)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

worst10 = sorted_idx[:10]
best10  = sorted_idx[-10:][::-1]

for ax, idxs, title, color in [
    (axes[0], worst10, 'En Düşük 10 Class', '#e74c3c'),
    (axes[1], best10,  'En Yüksek 10 Class', '#2ecc71')
]:
    labels = [f"[{i}] {NTU60_CLASSES[i][:22]}" for i in idxs]
    values = per_class_acc[idxs] * 100
    bars   = ax.barh(range(10), values, color=color, alpha=0.85)
    ax.set_yticks(range(10))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 105)
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

plt.suptitle('Best & Worst Action Classes', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'best_worst_classes.png'), dpi=130)
plt.show()

## 3. Confusion Matrix — En Çok Karıştırılan Pair'ler

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(18, 16))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
ax.set_xticks(range(60))
ax.set_yticks(range(60))
short = [f"{i}:{NTU60_CLASSES[i][:10]}" for i in range(60)]
ax.set_xticklabels(short, rotation=90, fontsize=5.5)
ax.set_yticklabels(short, fontsize=5.5)
ax.set_xlabel('Tahmin', fontsize=11)
ax.set_ylabel('Gerçek', fontsize=11)
ax.set_title(f'Normalized Confusion Matrix — Val Acc: {top1:.2f}%', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Off-diagonal en yüksek 15 hata pair'i
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

top_errors = sorted(
    [(cm_off[i,j], i, j) for i in range(60) for j in range(60) if cm_off[i,j] > 0],
    reverse=True
)[:15]

print(f"{'Gerçek':32s}  {'Tahmin':32s}  {'Hata':>5}  {'Recall':>6}")
print("-" * 80)
for cnt, true_c, pred_c in top_errors:
    recall = cm_off[true_c, pred_c] / cm[true_c].sum() * 100
    print(f"[{true_c:2d}]{NTU60_CLASSES[true_c]:28s}  "
          f"[{pred_c:2d}]{NTU60_CLASSES[pred_c]:28s}  "
          f"{int(cnt):5d}  {recall:5.1f}%")

## 4. Skeleton Görselleştirme Fonksiyonları

In [ ]:
def draw_skeleton_frame(ax, kp, title='', alpha=1.0, lw=2.0):
    """
    kp: (17, 2) — normalize edilmiş xy koordinatları
    """
    ax.clear()
    for (j1, j2), ec in EDGE_COLORS.items():
        ax.plot([kp[j1,0], kp[j2,0]], [kp[j1,1], kp[j2,1]],
                color=ec, linewidth=lw, alpha=alpha, solid_capstyle='round')
    for j in range(NUM_JOINTS):
        ax.scatter(kp[j,0], kp[j,1], c=JOINT_COLORS[j], s=40, zorder=5, alpha=alpha)
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')


def make_skeleton_gif(skeleton_seq, title, save_path, fps=10, person=0):
    """
    skeleton_seq: (M, T, 17, 2)  —  person=0 için person_idx
    Animasyonu GIF olarak kaydeder.
    """
    kp_seq = skeleton_seq[person]  # (T, 17, 2)
    # Sıfır olmayan frame'leri bul
    valid_frames = np.where(kp_seq.any(axis=(1,2)))[0]
    if len(valid_frames) == 0:
        valid_frames = np.arange(len(kp_seq))

    fig, ax = plt.subplots(figsize=(3.5, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#1a1a2e')

    def update(frame_idx):
        draw_skeleton_frame(ax, kp_seq[frame_idx],
                            title=f"{title}\nFrame {frame_idx+1}/{len(valid_frames)}",
                            lw=2.5)
        ax.set_facecolor('#1a1a2e')
        ax.title.set_color('white')

    ani = animation.FuncAnimation(fig, update, frames=valid_frames,
                                   interval=1000//fps, blit=False)
    ani.save(save_path, writer='pillow', fps=fps, dpi=80)
    plt.close(fig)
    return save_path


def show_static_frames(skeleton_seq, true_label, pred_label, confidence,
                       n_frames=8, person=0, save_path=None):
    """
    Bir sekans için N adet eşit aralıklı frame'i yan yana gösterir.
    """
    kp_seq = skeleton_seq[person]  # (T, 17, 2)
    valid  = np.where(kp_seq.any(axis=(1,2)))[0]
    if len(valid) == 0: valid = np.arange(len(kp_seq))

    frame_idxs = valid[np.linspace(0, len(valid)-1, n_frames, dtype=int)]

    correct = (true_label == pred_label)
    color   = '#2ecc71' if correct else '#e74c3c'
    verdict = 'DOĞRU' if correct else 'YANLIŞ'

    fig, axes = plt.subplots(1, n_frames, figsize=(n_frames*2.2, 3.5))
    fig.patch.set_facecolor('#1a1a2e')
    for i, (ax, fidx) in enumerate(zip(axes, frame_idxs)):
        ax.set_facecolor('#1a1a2e')
        draw_skeleton_frame(ax, kp_seq[fidx], title=f't={fidx}', lw=2)
        ax.title.set_color('white')

    suptitle = (f"{verdict} | Gerçek: [{true_label}] {NTU60_CLASSES[true_label]}\n"
                f"Tahmin: [{pred_label}] {NTU60_CLASSES[pred_label]}  "
                f"(güven: {confidence*100:.1f}%)")
    fig.suptitle(suptitle, fontsize=9, color=color, y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight',
                    facecolor='#1a1a2e')
    plt.show()
    plt.close()

print("Görselleştirme fonksiyonları hazır.")

## 5. Doğru Tahminler — Her Zorluktaki Düzeyden Örnekler

In [ ]:
# Doğru tahmin edilmiş, yüksek güvenli örnekler (kolay)
correct_idxs = np.where(correct_mask)[0]
conf_correct  = probs[correct_idxs, preds[correct_idxs]]
top_correct   = correct_idxs[np.argsort(conf_correct)[-6:][::-1]]

print("=== Yüksek Güvenli Doğru Tahminler ===")
for i, idx in enumerate(top_correct):
    true_c = y_val[idx]
    conf   = probs[idx, preds[idx]]
    print(f"  [{idx}] {NTU60_CLASSES[true_c]:35s} güven={conf*100:.1f}%")
    show_static_frames(
        X_val[idx], true_c, preds[idx], conf,
        save_path=os.path.join(VIZ_DIR, f'correct_{i}_{NTU60_CLASSES[true_c][:15].replace("/","-")}.png')
    )

## 6. Yanlış Tahminler — Hata Analizi

In [ ]:
# En çok karıştırılan 6 pair için birer örnek
top6_errors = top_errors[:6]

print("=== Yanlış Tahmin Örnekleri ===")
for rank, (cnt, true_c, pred_c) in enumerate(top6_errors):
    # Bu pair'den bir örnek bul
    candidate_idxs = np.where((y_val == true_c) & (preds == pred_c))[0]
    if len(candidate_idxs) == 0:
        continue
    # En yüksek güvenli yanlış tahmini seç (model en çok bu konuda yanılıyor)
    confs   = probs[candidate_idxs, pred_c]
    best_idx = candidate_idxs[np.argmax(confs)]
    conf    = probs[best_idx, pred_c]

    print(f"\n  Pair {rank+1}: [{true_c}]{NTU60_CLASSES[true_c]} → [{pred_c}]{NTU60_CLASSES[pred_c]}")
    fname = f'error_{rank+1}_{NTU60_CLASSES[true_c][:12].replace("/","-")}_as_{NTU60_CLASSES[pred_c][:12].replace("/","-")}.png'
    show_static_frames(
        X_val[best_idx], true_c, pred_c, conf,
        save_path=os.path.join(VIZ_DIR, fname)
    )

## 7. Skeleton Animasyonu (GIF)

In [ ]:
# Her hata tipinden birer GIF yap — rapor için
gif_pairs = top_errors[:4]

for rank, (cnt, true_c, pred_c) in enumerate(gif_pairs):
    candidate_idxs = np.where((y_val == true_c) & (preds == pred_c))[0]
    if len(candidate_idxs) == 0:
        continue
    best_idx = candidate_idxs[np.argmax(probs[candidate_idxs, pred_c])]

    title = (f"Gerçek: {NTU60_CLASSES[true_c]}\n"
             f"Tahmin: {NTU60_CLASSES[pred_c]}")
    gif_path = os.path.join(VIZ_DIR,
                            f'anim_{rank+1}_{NTU60_CLASSES[true_c][:12].replace("/","-")}.gif')
    make_skeleton_gif(X_val[best_idx], title, gif_path, fps=8)
    print(f"GIF kaydedildi: {gif_path}")
    display(Image(gif_path))

In [ ]:
# Ayrıca en yüksek acc'li 2 class için de GIF yap (doğru tahminler)
for rank, cls_id in enumerate(best10[:2]):
    idxs = np.where((y_val == cls_id) & (preds == cls_id))[0]
    if len(idxs) == 0:
        continue
    conf    = probs[idxs, cls_id]
    best_idx = idxs[np.argmax(conf)]

    title    = f"DOĞRU: {NTU60_CLASSES[cls_id]}"
    gif_path = os.path.join(VIZ_DIR, f'correct_anim_{NTU60_CLASSES[cls_id][:15].replace("/","-")}.gif')
    make_skeleton_gif(X_val[best_idx], title, gif_path, fps=8)
    print(f"GIF: {gif_path}")
    display(Image(gif_path))

## 8. Karıştırılan Pair — Yan Yana Karşılaştırma
En çok karıştırılan pair'in iki örneğini yan yana göster.

In [ ]:
_, true_c, pred_c = top_errors[0]

# Gerçek class'tan doğru tahmin edilmiş örnek
correct_of_true  = np.where((y_val == true_c) & (preds == true_c))[0]
# Gerçek class'tan yanlış tahmin edilmiş örnek
wrong_of_true    = np.where((y_val == true_c) & (preds == pred_c))[0]

if len(correct_of_true) > 0 and len(wrong_of_true) > 0:
    idx_ok   = correct_of_true[np.argmax(probs[correct_of_true, true_c])]
    idx_fail = wrong_of_true[np.argmax(probs[wrong_of_true, pred_c])]

    N_FRAMES = 6
    fig, axes = plt.subplots(2, N_FRAMES, figsize=(N_FRAMES*2.2, 7))
    fig.patch.set_facecolor('#1a1a2e')

    for row, (idx, label, color, verdict) in enumerate([
        (idx_ok,   f"[DOĞRU] {NTU60_CLASSES[true_c]}",  '#2ecc71', 'ok'),
        (idx_fail, f"[YANLIŞ→{NTU60_CLASSES[pred_c]}] Gerçek: {NTU60_CLASSES[true_c]}", '#e74c3c', 'fail'),
    ]):
        kp_seq = X_val[idx][0]  # person 0, (T,17,2)
        valid  = np.where(kp_seq.any(axis=(1,2)))[0]
        if len(valid) == 0: valid = np.arange(len(kp_seq))
        frame_idxs = valid[np.linspace(0, len(valid)-1, N_FRAMES, dtype=int)]

        for col, (ax, fidx) in enumerate(zip(axes[row], frame_idxs)):
            ax.set_facecolor('#1a1a2e')
            draw_skeleton_frame(ax, kp_seq[fidx], title=f't={fidx}', lw=2)
            ax.title.set_color('white')
            if col == 0:
                ax.set_ylabel(label, color=color, fontsize=8, labelpad=4)

    plt.suptitle(
        f'Karşılaştırma: {NTU60_CLASSES[true_c]} vs {NTU60_CLASSES[pred_c]}',
        fontsize=11, color='white', y=1.01
    )
    plt.tight_layout()
    save_path = os.path.join(VIZ_DIR, 'comparison_hardest_pair.png')
    plt.savefig(save_path, dpi=120, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    plt.close()
    print(f"Kaydedildi: {save_path}")

## 9. Özet — Kaydedilen Görseller

In [ ]:
saved = sorted(os.listdir(VIZ_DIR))
print(f"Drive/{VIZ_DIR.split('MyDrive/')[-1]}/ klasöründe {len(saved)} dosya:")
for f in saved:
    size = os.path.getsize(os.path.join(VIZ_DIR, f)) / 1024
    print(f"  {f:60s}  {size:6.1f} KB")

print(f"\nVal Top-1 Accuracy: {top1:.2f}%")
print(f"Kaydedilen görseller raporun Bölüm 4 (Qualitative Results) kısmı için kullanılabilir.")